# RA-PDS-KD Complete Colab Pipeline

**Reliability-Aware Participant-Disjoint Segment-Level Knowledge Distillation**

This is the only active notebook. Run cells top-to-bottom in one Colab runtime. TRAIN/DEV are used for teachers, KD, three student variants, and DEV robustness analysis. TEST remains closed until the explicit final-test gate at the end, then only CLEAN TEST is evaluated.


## 0. Clone/update repo and install once


In [ ]:
from pathlib import Path
import base64, subprocess, sys, runpy, os, json
from google.colab import userdata

REPO='engrkhubabahmad/reliability-aware-depression-kd'; BRANCH='feature/sbt-unimodal-teachers'
ROOT=Path('/content/reliability-aware-depression-kd')
token=userdata.get('GITHUB_TOKEN'); assert token, 'Add GITHUB_TOKEN in Colab Secrets.'
auth=base64.b64encode(f'x-access-token:{token}'.encode()).decode()
git=['git','-c',f'http.extraHeader=AUTHORIZATION: basic {auth}']
if not (ROOT/'.git').exists():
    subprocess.run(git+['clone','--branch',BRANCH,'--single-branch',f'https://github.com/{REPO}.git',str(ROOT)],check=True)
else:
    subprocess.run(git+['-C',str(ROOT),'fetch','origin',BRANCH],check=True)
    subprocess.run(['git','-C',str(ROOT),'checkout','-B',BRANCH,f'origin/{BRANCH}'],check=True)
subprocess.check_call([sys.executable,'-m','pip','install','-q','-r',str(ROOT/'requirements-colab.txt')])
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
print('RA-PDS-KD branch ready:',BRANCH)
print(subprocess.check_output(['git','-C',str(ROOT),'rev-parse','--abbrev-ref','HEAD'],text=True).strip())


## 1. SBT-Net unimodal teachers

Train the audio teacher from Wav2Vec2 and the text teacher from ALBERT-large using TRAIN participants only, select checkpoints on DEV, then freeze them and export logits/probabilities for the existing aligned RA-PDS-KD TRAIN/DEV segment IDs. Teacher-training chunks may differ from KD segments; TEST remains closed. Valid teacher checkpoints are reused automatically.


In [ ]:
script=ROOT/'scripts/teachers/sbt_unimodal/train.py'
assert script.exists(), f'Missing {script}. Current branch: '+subprocess.check_output(['git','-C',str(ROOT),'rev-parse','--abbrev-ref','HEAD'],text=True).strip()
subprocess.run([sys.executable,'-m','py_compile',str(script)],check=True)
_=runpy.run_path(str(script),run_name='__main__')


## 2. Build aligned TRAIN/DEV teacher targets


In [ ]:
script=ROOT/'scripts/kd/build_reliability_targets.py'
subprocess.run([sys.executable,'-m','py_compile',str(script)],check=True)
_=runpy.run_path(str(script),run_name='__main__')


## 3. Prepare lightweight student segment features

TRAIN/DEV only. Vocabulary and audio normalization are fitted on TRAIN only. Clean + deterministic noisy variants are prepared; missing modalities are generated during student training by zeroing that modality.


In [ ]:
script=ROOT/'scripts/students/prepare_segment_features.py'
subprocess.run([sys.executable,'-m','py_compile',str(script)],check=True)
_=runpy.run_path(str(script),run_name='__main__')


## 4. Train the three students + DEV robustness

1. No KD: clean TRAIN, hard labels only.  
2. Standard KD: clean TRAIN, hard labels + equal audio/text teacher targets.  
3. Reliability-Aware Robust KD: clean + audio missing + text missing + audio noisy + text noisy on TRAIN.  
All three are evaluated on DEV under the five conditions. Checkpoint/threshold selection uses CLEAN DEV only.


In [ ]:
script=ROOT/'scripts/students/train_segment_students.py'
subprocess.run([sys.executable,'-m','py_compile',str(script)],check=True)
_=runpy.run_path(str(script),run_name='__main__')


## 5. Review DEV results before opening TEST


In [ ]:
from pathlib import Path
import pandas as pd, json
DATA_ROOT=Path('/content/drive/MyDrive/DAIC_WOZ'); SEED=103
SROOT=DATA_ROOT/'experiments'/'students'/'segment_level_v1'/f'seed_{SEED}'
display(pd.read_csv(SROOT/'student_comparison_clean_dev.csv'))
for mode in ['no_kd','standard_kd','ra_robust_kd']:
    print('\n',mode)
    display(pd.read_csv(SROOT/mode/'dev_robustness_metrics.csv'))
print('\nCurrent frozen selection:')
print(json.dumps(json.loads((SROOT/'final_selection.json').read_text()),indent=2))


## 6. FINAL CLEAN TEST — locked

Do **not** run until the final student/checkpoint and DEV threshold are accepted. TEST is clean-only. No missing/noisy TEST conditions, no teacher query, no threshold search, no refitting.


In [ ]:
CONFIRM_FINAL_TEST=False
TEST_LABELS_CSV=''  # Optional legitimate final-stage CSV with participant_id,phq8_binary if official TEST split has no labels.
assert CONFIRM_FINAL_TEST, 'TEST CLOSED. Set CONFIRM_FINAL_TEST=True only for the one final clean evaluation.'
os.environ['RAPDSKD_OPEN_FINAL_TEST']='YES'
if TEST_LABELS_CSV: os.environ['TEST_LABELS_CSV']=TEST_LABELS_CSV
else: os.environ.pop('TEST_LABELS_CSV',None)
script=ROOT/'scripts/students/evaluate_final_clean_test.py'
subprocess.run([sys.executable,'-m','py_compile',str(script)],check=True)
_=runpy.run_path(str(script),run_name='__main__')
